In [1]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
from minatar import Environment 
import cv2
from stable_baselines3 import DQN
from sb3_contrib import QRDQN, TQC
from stable_baselines3.common.evaluation import evaluate_policy

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


In [21]:
import os
from IPython.display import HTML
from base64 import b64encode
from pyvirtualdisplay import Display
import numpy as np
import imageio
from IPython.display import Video, display

In [3]:
def render_mp4(videopath: str) -> HTML:
    mp4 = open(videopath, "rb").read()
    encoded = b64encode(mp4).decode()
    return HTML(
        f"""
    <video width="420" height="420" controls>
      <source src="data:video/mp4;base64,{encoded}" type="video/mp4">
    </video>
    """
    )

In [4]:
def minatar_obs_to_rgb(obs):
    # obs chega 3D no vídeo
    obs_uint8 = (obs * 255).astype(np.uint8)

    # soma canais para formar imagem
    gray = obs_uint8.sum(axis=2)
    gray = np.clip(gray, 0, 255).astype(np.uint8)

    rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
    rgb = cv2.resize(rgb, (300, 300), interpolation=cv2.INTER_NEAREST)
    return rgb

In [5]:
class MinAtarEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, game="breakout"):
        super().__init__()
        self.env = Environment(env_name=game)

        obs_shape = self.env.state_shape()
        self.observation_space = spaces.Box(
            low=0, high=1, shape=obs_shape, dtype=np.float32
        )

        self.action_space = spaces.Discrete(self.env.num_actions())

    def reset(self, seed=None, options=None):
        if seed is not None:
            np.random.seed(seed)
        self.env.reset()
        return self.env.state(), {}

    def step(self, action):
        reward, done = self.env.act(action)
        return self.env.state(), reward, done, False, {}

    def render(self):
        self.env.display_state()

In [6]:
class BlurWrapper(gym.ObservationWrapper):
    def __init__(self, env, blur=True, region=(0, 0, 5, 5), ksize=3):
        super().__init__(env)
        self.blur = blur
        self.x1, self.y1, self.x2, self.y2 = region
        self.ksize = ksize

    def observation(self, obs):
        if not self.blur:
            return obs

        obs = obs.copy()

        # seleção da região
        patch = obs[self.y1 : self.y2, self.x1 : self.x2, :]

        # converter de bool para float32 (obrigatório para o blur)
        patch = patch.astype(np.float32)

        # aplicar blur
        patch = cv2.GaussianBlur(patch, (self.ksize, self.ksize), 0)

        # voltar para float32 com faixa 0–1
        patch = np.clip(patch, 0, 1)

        # recolocar no frame
        obs[self.y1 : self.y2, self.x1 : self.x2, :] = patch

        return obs

In [7]:
class FlattenWrapper(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        shape = env.observation_space.shape
        flat = int(np.prod(shape))
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(flat,), dtype=np.float32
        )

    def observation(self, obs):
        return obs.flatten().astype(np.float32)

In [8]:
def make_breakout_env(blur=False):
    env = MinAtarEnv("breakout")
    env = BlurWrapper(env, blur=blur, region=(0, 0, 10, 4), ksize=3)
    env = FlattenWrapper(env)
    return env

In [9]:
def make_breakout_env_for_video(blur=False):
    env = MinAtarEnv("breakout")
    env = BlurWrapper(env, blur=blur, region=(0, 0, 10, 4), ksize=3)
    return env  # sem FLATTEN

In [10]:
def build_model(algo, env):
    if algo == "dqn":
        return DQN("MlpPolicy", env, verbose=1)
    elif algo == "rainbow":
        return QRDQN("MlpPolicy", env, verbose=1)
    elif algo == "c51":
        return DQN("MlpPolicy", env, verbose=1, policy_kwargs=dict(n_atoms=51))
    elif algo == "sac_discreto":
        return TQC("MlpPolicy", env, verbose=1)
    else:
        raise ValueError("Algoritmo inválido.")

In [11]:
def run_experiment_breakout(algo="dqn", blur=False):

    env = make_breakout_env(blur=blur)
    model = build_model(algo, env)

    print(f"\nTreinando {algo} | blur={blur}")
    model.learn(total_timesteps=50_000)

    mean_reward, std = evaluate_policy(model, env, n_eval_episodes=20)
    print(f"Reward médio: {mean_reward}")

    return model, env, mean_reward

In [25]:
def _obs_to_frame_minatar(obs: np.ndarray) -> np.ndarray:
    """
    Converte uma observação do MinAtar (achatada ou não) em frame RGB (H, W, 3) uint8.
    Trata shapes:
      - (400,)  -> assume grade 10x10 com C canais (C = 400 / 100)
      - (H, W)
      - (H, W, C)
    """
    obs = np.array(obs)

    # Caso 1: vetor achatado (ex.: (400,))
    if obs.ndim == 1:
        grid = 10  # MinAtar usa 10x10
        size = obs.size
        if size % (grid * grid) != 0:
            raise ValueError(
                f"Não consegui fatorar obs achatado {obs.shape} para 10x10xC."
            )
        channels = size // (grid * grid)
        obs = obs.reshape(grid, grid, channels)

    # Caso 2: (H, W) -> adiciona eixo de canal
    elif obs.ndim == 2:
        obs = obs[..., None]  # (H, W, 1)

    # Caso 3: (H, W, C) já ok
    elif obs.ndim == 3:
        pass
    else:
        raise ValueError(f"Obs com shape inesperado: {obs.shape}")

    # Agora obs está (H, W, C)
    h, w, c = obs.shape

    if c == 1:
        frame = np.repeat(obs, 3, axis=-1)
    elif c >= 3:
        frame = obs[..., :3]
    else:  # c == 2
        pad = np.zeros((h, w, 3 - c), dtype=obs.dtype)
        frame = np.concatenate([obs, pad], axis=-1)

    # Normaliza para 0–255
    if frame.dtype != np.uint8:
        max_val = frame.max()
        if max_val == 0:
            frame = np.zeros_like(frame, dtype=np.uint8)
        else:
            frame = (frame / max_val * 255).astype(np.uint8)

    return frame

In [23]:
def record_and_display_minatar(
    model,
    blur: bool = False,  # mantido só pra compatibilidade com sua chamada
    video_name: str = "minatar_run",
    num_steps: int = 2000,  # quantos steps gravar
    fps: int = 15,  # FPS do vídeo
):
    """
    Grava a execução do modelo no próprio env do SB3 (MinAtar) e mostra o vídeo.
    Não depende de `gym.make` nem de `render()`.
    """

    # 1) Pega o env que o modelo usou no treino
    env = model.get_env()
    if env is None:
        raise ValueError(
            "model.get_env() retornou None. Passe um VecEnv explicitamente ou treine o modelo com env embutido."
        )

    print("Iniciando gravação...")

    # VecEnv.reset() retorna apenas obs (sem info)
    obs = env.reset()
    frames = []

    for step in range(num_steps):
        # obs é (n_envs, H, W, C). Pegamos o primeiro ambiente.
        obs_first = obs[0]
        frame = _obs_to_frame_minatar(obs_first)
        frames.append(frame)

        # 2) Ação do modelo
        action, _ = model.predict(obs, deterministic=True)

        # Para VecEnv (SB3): obs, rewards, dones, infos
        obs, rewards, dones, infos = env.step(action)

        # Se quiser parar depois de terminar o primeiro episódio:
        # if dones[0]:
        #     break

    print(f"Gravação finalizada. Frames gravados: {len(frames)}")

    # 3) Salva o vídeo
    os.makedirs("video", exist_ok=True)
    video_path = os.path.join("video", f"{video_name}.mp4")
    imageio.mimsave(video_path, frames, fps=fps)

    # 4) Mostra no notebook
    display(Video(video_path, embed=True))
    return video_path

In [13]:
model, env, reward = run_experiment_breakout("dqn", blur=False)

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.

Treinando dqn | blur=False
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 21       |
|    ep_rew_mean      | 1.5      |
|    exploration_rate | 0.984    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 11122    |
|    time_elapsed     | 0        |
|    total_timesteps  | 84       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 14.8     |
|    ep_rew_mean      | 0.875    |
|    exploration_rate | 0.978    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 398      |
|    time_elapsed     | 0        |
|    total_timesteps  | 118      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.0467   |
|    n_updates        | 4        |
--

/home/hartb/Estudos/BlurRL/.venv/lib/python3.11/site-packages/stable_baselines3/common/evaluation.py:70: UserWarning: Evaluation environment is not wrapped with a ``Monitor`` wrapper. This may result in reporting modified episode lengths and rewards, if other wrappers happen to modify these. Consider wrapping environment first with ``Monitor`` wrapper.
  warnings.warn(


In [27]:
record_and_display_minatar(model, blur=False, video_name="breakout_no_blur")

Iniciando gravação...


AcceleratorError: CUDA error: unspecified launch failure
Search for `cudaErrorLaunchFailure' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [28]:
model.save("dqn_minatar_breakout")
print("Modelo salvo!")

AcceleratorError: CUDA error: unspecified launch failure
Search for `cudaErrorLaunchFailure' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
import gc, torch

# libera o modelo antigo e a GPU
del model
gc.collect()
torch.cuda.empty_cache()

In [ ]:

from stable_baselines3 import DQN

# carrega o MESMO modelo, mas forçando usar CPU
model_cpu = DQN.load("dqn_minatar_breakout", device="cpu")
print("Modelo recarregado em:", model_cpu.device)

In [15]:
model, env, reward = run_experiment_breakout("dqn", blur=False)
record_and_display_minatar(model, blur=False, video_name="breakout_blur")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.

Treinando dqn | blur=False
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 6        |
|    ep_rew_mean      | 0        |
|    exploration_rate | 0.995    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 13402    |
|    time_elapsed     | 0        |
|    total_timesteps  | 24       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 9.75     |
|    ep_rew_mean      | 0.375    |
|    exploration_rate | 0.985    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 15134    |
|    time_elapsed     | 0        |
|    total_timesteps  | 78       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 10.2     |
| 